<a href="https://colab.research.google.com/github/OdysseusPolymetis/enssib_class/blob/main/spatialisation_qgis_carto_gephi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Du texte à la carte (et au réseau)

**Cartographie, QGIS, CARTO, Gephi**

Ce notebook part d'un texte littéraire et produit tout ce qu'il faut pour le cartographier :

| Partie | Ce qu'on fabrique | Outil |
|---|---|---|
| **A. Les lieux du texte** | les lieux cités, repérés automatiquement, nettoyés et géocodés | CARTO, QGIS |
| **B. Le trajet du récit** | le lieu où se trouve le récit, tranche par tranche, relié en trajet | CARTO, QGIS |
| **C. Un itinéraire de référence** | un itinéraire connu, daté dans le temps du récit | CARTO, QGIS |
| **D. Des réseaux** | lieux cités ensemble ; personnages et lieux | Gephi |
| **E. Lieux traversés, lieux évoqués** | la distance de chaque lieu au trajet | CARTO, QGIS |

Par défaut, le texte est *Le Tour du monde en quatre-vingts jours* (Jules Verne, 1872). **Pour travailler sur votre propre texte, il suffit de modifier la cellule de configuration** (partie 0).

Tous les fichiers produits sont écrits dans le dossier `donnees/`.

> Les cellules marquées **"À vous"** sont des réglages à adapter à votre texte.

## 0. Installation et configuration

À lancer **une seule fois**. Téléchargez le modèle spaCy de la langue de votre texte.

In [ ]:
!pip install requests pandas spacy geopy networkx folium matplotlib
!python -m spacy download fr_core_news_md     # texte en français
!python -m spacy download en_core_web_md      # texte en anglais

### Configuration : la seule cellule à modifier pour changer de texte

- `SOURCE_TEXTE` : une adresse web (par exemple un texte du Projet Gutenberg en « Plain Text UTF-8 ») **ou** le nom d'un fichier `.txt` déposé à côté du notebook, enregistré en UTF-8.
- `LANGUE` : `"fr"` ou `"en"`.
- `FICHIER_ETAPES` : **facultatif**. Un CSV placé dans `donnees/`, qui donne un itinéraire connu dans l'ordre, avec au minimum les colonnes `lieu,lat,lon` (et si possible `nom_texte`, l'orthographe du lieu dans le texte). Sans ce fichier, mettez `None` : les parties A4 et C sont sautées, tout le reste fonctionne.

In [ ]:
NOM_PROJET = "tour_du_monde"          # préfixe des fichiers produits
SOURCE_TEXTE = "https://www.gutenberg.org/ebooks/800.txt.utf-8"   # ou "mon_texte.txt"
LANGUE = "fr"                          # "fr" ou "en"
FICHIER_ETAPES = "etapes_fogg.csv"     # ou None si vous n'avez pas de fichier d'étapes

In [ ]:
import json
import re
import time
from pathlib import Path

import pandas as pd
import requests

DOSSIER = Path("donnees")
DOSSIER.mkdir(exist_ok=True)

MODELES = {
    # langue: (modèle spaCy, étiquettes qui désignent des lieux)
    "fr": ("fr_core_news_md", {"LOC"}),
    "en": ("en_core_web_md", {"GPE", "LOC", "FAC"}),
}
MODELE, LABELS_LIEUX = MODELES[LANGUE]


def fichier(suffixe):
    return DOSSIER / f"{NOM_PROJET}_{suffixe}"


# Itinéraire de référence (facultatif)
etapes = None
if FICHIER_ETAPES and (DOSSIER / FICHIER_ETAPES).exists():
    etapes = pd.read_csv(DOSSIER / FICHIER_ETAPES)
    if "nom_texte" not in etapes.columns:
        etapes["nom_texte"] = etapes["lieu"]
    if "ordre" not in etapes.columns:
        etapes["ordre"] = range(1, len(etapes) + 1)
    print(f"Itinéraire de référence : {len(etapes)} étapes")
elif FICHIER_ETAPES:
    print(f"{DOSSIER / FICHIER_ETAPES} introuvable : on continue sans itinéraire de référence")
else:
    print("Pas d'itinéraire de référence : les parties A4 et C seront sautées")
print("Modèle :", MODELE, "| étiquettes de lieux :", LABELS_LIEUX)

---
# Partie A — Les lieux du texte

## A1. Récupérer et préparer le texte

On lit le texte (téléchargé ou local), on retire l'en-tête et le pied de page du Projet Gutenberg s'il y en a, puis on le découpe en paragraphes.

Deux mises en forme sont reconnues :
- des paragraphes séparés par une **ligne vide**, avec des lignes coupées à ~70 caractères (format Gutenberg) ;
- **un paragraphe par ligne**, sans ligne vide (format courant d'un `.txt` exporté d'un traitement de texte).

In [ ]:
if str(SOURCE_TEXTE).startswith("http"):
    fichier_texte = fichier("texte.txt")
    if not fichier_texte.exists():
        r = requests.get(SOURCE_TEXTE, timeout=60)
        r.raise_for_status()
        r.encoding = "utf-8"
        fichier_texte.write_text(r.text, encoding="utf-8")
else:
    fichier_texte = Path(SOURCE_TEXTE)

brut = fichier_texte.read_text(encoding="utf-8")

# En-tête et pied de page Gutenberg (marqueurs *** START ... *** et *** END ... ***)
debut, fin = brut.find("*** START"), brut.find("*** END")
texte = brut[brut.find("\n", debut) + 1 : fin] if debut != -1 and fin != -1 else brut

texte = texte.replace("\r\n", "\n").replace("\u2019", "'")
texte = texte.replace("_", "")          # italiques Gutenberg : « _Tankadère_ »

lignes = texte.split("\n")
if texte.count("\n\n") < 0.05 * len(lignes):
    # pas de lignes vides : une ligne = un paragraphe
    paragraphes = [l.strip() for l in lignes if l.strip()]
else:
    texte = re.sub(r"-\n(?=\S)", "-", texte)          # « Hong-\nKong » -> « Hong-Kong »
    texte = re.sub(r"(?<!\n)\n(?!\n)", " ", texte)    # on recolle les lignes d'un paragraphe
    paragraphes = [p.strip() for p in texte.split("\n\n") if p.strip()]
texte = "\n\n".join(paragraphes)

print(f"{len(texte):,} caractères, {len(paragraphes):,} paragraphes")
print(paragraphes[min(5, len(paragraphes) - 1)][:300])

## A2. Repérer les entités nommées (REN / NER)

Le modèle de spaCy étiquette les entités : en français `PER` (personnes), `LOC` (lieux), `ORG` (organisations), `MISC` (divers) ; en anglais `PERSON`, `GPE` (pays, villes), `LOC`, `FAC` (bâtiments)…
On garde **toutes** les entités avec leur étiquette, et leur position dans le texte (en % des paragraphes), approximation du "temps du récit".

Environ une minute pour un roman.

In [ ]:
import spacy

nlp = spacy.load(MODELE)


def extraire_entites(nlp, paragraphes):
    lignes = []
    for i, doc in enumerate(nlp.pipe(paragraphes, batch_size=50)):
        for ent in doc.ents:
            lignes.append({"texte": ent.text, "label": ent.label_, "paragraphe": i})
    df = pd.DataFrame(lignes, columns=["texte", "label", "paragraphe"])
    df["position_pct"] = (df["paragraphe"] / len(paragraphes) * 100).round(1)
    return df


entites = extraire_entites(nlp, paragraphes)
print(len(entites), "entités, dont", entites["label"].isin(LABELS_LIEUX).sum(), "étiquetées comme lieux")
entites["label"].value_counts()

## A3. Normaliser les noms

La REN renvoie le texte tel qu'il apparaît : "la Chine" et "Chine", "INDE" (un titre de chapitre) et "Inde" seraient comptés à part. La fonction `normaliser` retire la ponctuation et les **articles en minuscule** en tête ("la Chine" --> "Chine" ; "Le Havre" garde son article), et remet en casse normale les noms tout en majuscules ou tout en minuscules.

On regroupe ensuite les variantes sur une **clé** (minuscules, tirets remplacés par des espaces) : "Hong-Kong" = "Hong Kong".

In [ ]:
DETERMINANTS = {
    # un article ou une préposition en minuscule, suivi d'une espace ou d'une apostrophe
    "fr": r"^(?:(?:de la|le|la|les|du|des|de|au|aux|en)\s+|(?:l'|d'))+",
    "en": r"^(?:the)\s+",
}[LANGUE]


def normaliser(nom):
    nom = nom.replace("_", "").replace("\u2019", "'")
    nom = re.sub(r"\s+", " ", nom).strip(" .,;:!?«»\"'()[]-")
    nom = re.sub(DETERMINANTS, "", nom)          # sensible à la casse : "Le Havre" reste
    if nom.isupper() and len(nom) > 3:
        nom = nom.title()                         # INDE -> Inde
    elif nom.islower():
        nom = nom[:1].upper() + nom[1:]           # paris -> Paris
    return nom


def cle(nom):
    return re.sub(r"[-\s]+", " ", nom.lower()).strip()


def motif(nom):
    # expression régulière qui trouve le nom, avec tiret ou espace ("New-York" / "New York")
    return r"\b" + re.sub(r"\\[-\s]|\\ ", r"[-\\s]", re.escape(nom)) + r"\b"


for exemple in ["la Chine", "du Japon", "INDE", "paris", "Hong-Kong", "Le Havre"]:
    print(f"{exemple!r:12} -> {normaliser(exemple)!r:12} clé : {cle(normaliser(exemple))!r}")

## A4. Diagnostic : qu'est-ce que la REN a manqué ? *(si itinéraire de référence)*

Pour chaque étape de l'itinéraire de référence, on compte ses occurrences dans le texte, les fois où la REN l'a étiquetée comme lieu, et les fois où elle l'a vue **avec une autre étiquette**. C'est une mesure du **rappel**.

Sans itinéraire de référence, faites le même contrôle à la main avec `ou_est("…")` (partie A6) sur quelques lieux que vous savez présents.

In [ ]:
def diagnostic(entites, noms):
    ent = entites.assign(cle=entites["texte"].map(normaliser).map(cle))
    lignes = []
    for nom in dict.fromkeys(noms):
        trouve = ent[ent["cle"] == cle(nom)]
        autres = trouve[~trouve["label"].isin(LABELS_LIEUX)]["label"].value_counts()
        lignes.append({
            "lieu": nom,
            "dans_le_texte": len(re.findall(motif(nom), texte, flags=re.IGNORECASE)),
            "comme_lieu": int(trouve["label"].isin(LABELS_LIEUX).sum()),
            "autres_etiquettes": ", ".join(f"{lab} ×{n}" for lab, n in autres.items()),
        })
    d = pd.DataFrame(lignes)
    d["rappel_pct"] = (100 * d["comme_lieu"] / d["dans_le_texte"].where(d["dans_le_texte"] > 0)).round(0)
    return d


if etapes is not None:
    display(diagnostic(entites, etapes["nom_texte"]))
else:
    print("Pas d'itinéraire de référence : diagnostic sauté.")

## A5. Aider le modèle avec un dictionnaire de lieux

Le composant `entity_ruler`, placé avant le modèle statistique, étiquette comme lieu toute occurrence d'un nom de la liste. Le dictionnaire garantit les lieux **qu'on connaît déjà** ; la REN sert à découvrir **ceux qu'on ne connaît pas**.

**À vous** : ajoutez à `AUTRES_LIEUX` les lieux que le modèle rate (orthographe du texte). Les étapes de l'itinéraire de référence, s'il existe, sont ajoutées automatiquement.

In [ ]:
AUTRES_LIEUX = [
    # à compléter, ex. "Saint-Martin-des-Brumes"
]


def variantes(nom):
    return {nom, nom.replace("-", " "), nom.replace(" ", "-")}


noms_connus = (list(etapes["nom_texte"]) if etapes is not None else []) + AUTRES_LIEUX
if noms_connus:
    label_dictionnaire = "LOC" if LANGUE == "fr" else "GPE"
    nlp_regles = spacy.load(MODELE)
    ruler = nlp_regles.add_pipe("entity_ruler", before="ner", config={"phrase_matcher_attr": "LOWER"})
    ruler.add_patterns([{"label": label_dictionnaire, "pattern": v} for nom in noms_connus for v in variantes(nom)])
    entites = extraire_entites(nlp_regles, paragraphes)
    print(len(entites), "entités, dont", entites["label"].isin(LABELS_LIEUX).sum(), "étiquetées comme lieux")
    display(diagnostic(entites, noms_connus))
else:
    print("Dictionnaire vide : on garde les entités de A2.")

## A6. Nettoyer et compter

Il reste des faux positifs : personnages, navires, formules, et des **gentilés** ("Américains", "Français") pris pour des lieux.

**À vous** : parcourez la liste produite et complétez `A_EXCLURE` et `GENTILES`. Les valeurs fournies sont celles du *Tour du monde* : **remplacez-les pour votre texte**.

`MIN_MENTIONS` fixe le nombre minimal de mentions pour garder un lieu.

In [ ]:
A_EXCLURE = {
    # exemples pour le Tour du monde : personnages, navires, faux positifs
    "Passepartout", "Fogg", "Phileas Fogg", "Fix", "Aouda",
    "Mongolia", "Rangoon", "Carnatic", "Tankadère", "Henrietta",
    "Mrs", "Esq", "Croyez", "Honneur", "Soleil", "Railway", "Palki-ghari",
    # à compléter
}
GENTILES = {
    "Américain", "Américains", "Français", "Anglais", "Chinois", "Japonais",
    "Européen", "Européens", "Européenne", "Parisien", "Indien", "Indiens", "Pawnies",
    # à compléter
}
FUSIONS = {
    "Havre": "Le Havre",
    # à compléter, ex. "Céleste Empire": "Chine"
}
MIN_MENTIONS = 1

mentions = (
    entites[entites["label"].isin(LABELS_LIEUX)]
    .assign(lieu=lambda d: d["texte"].map(normaliser).replace(FUSIONS))
    .loc[lambda d: ~d["lieu"].isin(A_EXCLURE | GENTILES) & (d["lieu"].str.len() > 2)]
    .assign(cle=lambda d: d["lieu"].map(cle))
)
forme_affichee = mentions.groupby("cle")["lieu"].agg(lambda s: s.value_counts().index[0])

lieux_propres = (
    mentions.groupby("cle")
    .agg(nb_mentions=("lieu", "size"),
         premiere_mention_pct=("position_pct", "min"),
         derniere_mention_pct=("position_pct", "max"))
    .assign(lieu=forme_affichee)
    .reset_index(drop=True)
    .query("nb_mentions >= @MIN_MENTIONS")
    .sort_values("nb_mentions", ascending=False)
    .reset_index(drop=True)
    [["lieu", "nb_mentions", "premiere_mention_pct", "derniere_mention_pct"]]
)
print(len(mentions), "mentions,", len(lieux_propres), "lieux retenus")
lieux_propres.head(30)

### Où est passé ce lieu ?

Le tableau n'affiche que les 30 lieux les plus cités. `ou_est("…")` suit un lieu à travers toute la chaîne (texte --> entités --> liste finale --> géocodage).

In [ ]:
def ou_est(nom):
    k = cle(normaliser(nom))
    print(f"« {nom} »")
    print(f"  dans le texte : {len(re.findall(motif(nom), texte, flags=re.IGNORECASE))} occurrence(s)")
    ent = entites.assign(cle=entites["texte"].map(normaliser).map(cle))
    vues = ent[ent["cle"].str.contains(k, regex=False)].groupby(["texte", "label"]).size()
    print("  entités repérées :", ", ".join(f"« {t} » {lab} ×{n}" for (t, lab), n in vues.items()) or "aucune")
    rangs = lieux_propres.reset_index().loc[lambda d: d["lieu"].map(cle).str.contains(k, regex=False)]
    if rangs.empty:
        print("  absent de lieux_propres : pas étiqueté comme lieu, exclu (A_EXCLURE, GENTILES) ou sous MIN_MENTIONS")
    limite = globals().get("N_MAX", 60)
    for _, r in rangs.iterrows():
        rang = r["index"] + 1
        print(f"  dans lieux_propres : « {r['lieu']} », rang {rang} sur {len(lieux_propres)}, {r['nb_mentions']} mention(s)")
        if rang > limite:
            print(f" ! rang > N_MAX ({limite}) : il ne sera pas géocodé")


ou_est(lieux_propres["lieu"].iloc[0])

## A7. Géocoder

Géocoder = transformer un **nom** en **coordonnées**. On utilise Nominatim (OpenStreetMap) : **1 requête par seconde maximum**, et un `user_agent` qui vous identifie. Les résultats sont **mis en cache** dans un fichier JSON : relancer la cellule ne redemande pas les lieux déjà trouvés.

On garde aussi le **type de lieu** renvoyé par Nominatim (`city`, `country`, `state`…) : il servira à construire le trajet du récit (partie B).

In [ ]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="master-hn-tp-carto (remplacez par votre e-mail)", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

CHEMIN_CACHE = DOSSIER / "cache_geocodage.json"
cache = json.loads(CHEMIN_CACHE.read_text(encoding="utf-8")) if CHEMIN_CACHE.exists() else {}


def geocoder(nom):
    # Renvoie (lat, lon, adresse trouvée, type de lieu) ou des None
    if nom not in cache:
        try:
            loc = geocode(nom, language=LANGUE)
        except Exception as e:
            print("Erreur pour", nom, ":", e)
            return None, None, None, None
        if loc:
            type_lieu = loc.raw.get("addresstype") or loc.raw.get("type")
            cache[nom] = [loc.latitude, loc.longitude, loc.address, type_lieu]
        else:
            cache[nom] = None
        CHEMIN_CACHE.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")
    if not cache[nom]:
        return None, None, None, None
    return tuple((list(cache[nom]) + [None])[:4])    # anciens caches : pas de type

In [ ]:
# Seuls les N_MAX lieux les plus cités sont géocodés (120 lieux environ 2 min).
# Pour tout géocoder : N_MAX = len(lieux_propres). Le cache évite de refaire les requêtes.
N_MAX = 120

a_geocoder = lieux_propres.head(N_MAX).copy()
resultats = [geocoder(nom) for nom in a_geocoder["lieu"]]
a_geocoder[["lat", "lon", "adresse_trouvee", "type_lieu"]] = pd.DataFrame(resultats, index=a_geocoder.index)

print(a_geocoder["lat"].notna().sum(), "/", len(a_geocoder), "lieux géocodés")
a_geocoder.head(20)

**À vous** : lisez `adresse_trouvee`. Nominatim a-t-il trouvé le bon lieu ? Un nom ancien ou ambigu peut tomber ailleurs ("Queenstown" en Nouvelle-Zélande plutôt qu'en Irlande). Corrigez dans `CORRECTIONS`, avec des coordonnées prises sur OpenStreetMap ou Wikidata.

In [ ]:
CORRECTIONS = {
    # "Nom du lieu": (latitude, longitude, "type"),   type : "city", "country", "water"…
    # à compléter
}
for nom, (lat, lon, type_lieu) in CORRECTIONS.items():
    a_geocoder.loc[a_geocoder["lieu"] == nom, ["lat", "lon", "adresse_trouvee", "type_lieu"]] = [
        lat, lon, "correction manuelle", type_lieu]

lieux_geo = a_geocoder.dropna(subset=["lat", "lon"]).reset_index(drop=True)
# colonne commune à toutes les couches : un seul filtre CARTO pourra agir sur toutes
lieux_geo["position_pct"] = lieux_geo["premiere_mention_pct"]
lieux_geo["cle"] = lieux_geo["lieu"].map(cle)
print(len(lieux_geo), "lieux géocodés")
lieux_geo["type_lieu"].value_counts()

## A8. Exporter

CSV avec colonnes `lat`/`lon`, et **GeoJSON**, le format vectoriel du web (ordre **longitude, latitude**). Ces fichiers sont réécrits en partie E avec la distance de chaque lieu au trajet.

In [ ]:
def _json_safe(v):
    if hasattr(v, "item"):
        v = v.item()
    if isinstance(v, float) and v != v:
        return None
    return v


def points_vers_geojson(df, chemin, lat="lat", lon="lon"):
    features = []
    for _, row in df.dropna(subset=[lat, lon]).iterrows():
        props = {k: _json_safe(v) for k, v in row.items() if k not in (lat, lon, "cle")}
        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [float(row[lon]), float(row[lat])]},
            "properties": props,
        })
    Path(chemin).write_text(
        json.dumps({"type": "FeatureCollection", "features": features}, ensure_ascii=False, indent=1),
        encoding="utf-8")
    print(f"Écrit : {chemin} ({len(features)} points)")


def exporter_lieux():
    lieux_geo.drop(columns=["cle"]).to_csv(fichier("lieux.csv"), index=False, encoding="utf-8")
    points_vers_geojson(lieux_geo, fichier("lieux.geojson"))


exporter_lieux()

## A9. Vérifier les points aberrants

**Fond de carte** : on utilise un fond Esri, sans clé API. La carte se cadre sur l'ensemble des points : si elle est très dézoomée, un point est parti loin.

In [ ]:
import folium

FOND = "Esri.WorldGrayCanvas"   # autres fonds sans clé : "Esri.WorldStreetMap", "Esri.WorldImagery"
carte = folium.Map(tiles=FOND)
for _, l in lieux_geo.iterrows():
    folium.CircleMarker(
        [l["lat"], l["lon"]],
        radius=3 + l["nb_mentions"] ** 0.5,
        tooltip=l["lieu"],
        popup=f"{l['lieu']} ({l['nb_mentions']} mentions)<br>{l['adresse_trouvee']}",
        color="#B5472F",
        fill=True,
    ).add_to(carte)
carte.fit_bounds(lieux_geo[["lat", "lon"]].values.tolist())
carte

**Plan B, sans fond de carte ni réseau** : un nuage de points en longitude/latitude (et l'itinéraire de référence en gris, s'il existe).

In [ ]:
import matplotlib.pyplot as plt


def couper_antimeridien(lons, lats):
    x, y = [], []
    for i, (lo, la) in enumerate(zip(lons, lats)):
        if i and abs(lo - x[-1]) > 180:
            x.append(float("nan"))
            y.append(float("nan"))
        x.append(lo)
        y.append(la)
    return x, y


fig, ax = plt.subplots(figsize=(14, 7))
if etapes is not None:
    ax.plot(*couper_antimeridien(etapes["lon"], etapes["lat"]), "-o", color="#B9B2A3",
            markersize=4, label="itinéraire de référence")
ax.scatter(lieux_geo["lon"], lieux_geo["lat"], s=10 + 4 * lieux_geo["nb_mentions"],
           color="#B5472F", alpha=0.7, label="lieux géocodés")
for _, l in lieux_geo.iterrows():
    ax.annotate(l["lieu"], (l["lon"], l["lat"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set(xlim=(-180, 180), ylim=(-60, 80), xlabel="longitude", ylabel="latitude")
ax.grid(alpha=0.3)
ax.legend(loc="lower left")
plt.show()

---
# Partie B — Le trajet du récit

Où se trouve le récit, au fil de la lecture ? On découpe le texte en `N_TRANCHES` tranches (50 tranches = 2 % du texte chacune) et on retient, pour chaque tranche, **le lieu le plus cité**. Les tranches successives dominées par le même lieu sont fusionnées. En reliant ces lieux dans l'ordre, on obtient le **trajet du récit**.

Cela marche pour n'importe quel texte, sans connaître l'itinéraire à l'avance. Pour un roman de voyage, le trajet du récit retrouve à peu près l'itinéraire ; pour un autre texte, il montre les lieux où le récit s'installe, et dans quel ordre.

On écarte les grands ensembles (pays, régions) avec `TYPES_EXCLUS` : sinon "Inde" ou "Angleterre" dominent des tranches entières.

**À vous** : le trajet automatique fait-il des détours étranges ? Un lieu seulement évoqué (un souvenir, une comparaison) peut dominer une tranche où rien d'autre n'est cité. Montez `MIN_MENTIONS_TRANCHE` à 2 ou 3, ou excluez ce lieu.

In [ ]:
N_TRANCHES = 50
TYPES_EXCLUS = {"country", "state", "region", "province", "county", "continent"}
MIN_MENTIONS_TRANCHE = 1   # 2 ou 3 : un lieu cité une seule fois ne peut plus « dominer » une tranche

geo = lieux_geo[~lieux_geo["type_lieu"].isin(TYPES_EXCLUS)][["cle", "lieu", "lat", "lon", "type_lieu"]]
m = mentions.merge(geo.rename(columns={"lieu": "lieu_geo"}), on="cle")
m["tranche"] = (m["position_pct"] // (100 / N_TRANCHES)).astype(int).clip(upper=N_TRANCHES - 1)

dominants = (
    m.groupby(["tranche", "lieu_geo"]).size().reset_index(name="n")
    .sort_values(["tranche", "n"], ascending=[True, False])
    .drop_duplicates("tranche")
    .loc[lambda d: d["n"] >= MIN_MENTIONS_TRANCHE]
)
dominants["bloc"] = (dominants["lieu_geo"] != dominants["lieu_geo"].shift()).cumsum()
trajet = (
    dominants.groupby("bloc")
    .agg(lieu=("lieu_geo", "first"), tranche_debut=("tranche", "min"),
         tranche_fin=("tranche", "max"), mentions=("n", "sum"))
    .reset_index(drop=True)
)
trajet["debut_pct"] = trajet["tranche_debut"] * 100 / N_TRANCHES
trajet["fin_pct"] = (trajet["tranche_fin"] + 1) * 100 / N_TRANCHES
trajet = trajet.merge(geo.drop_duplicates("lieu")[["lieu", "lat", "lon", "type_lieu"]], on="lieu", how="left")
trajet.insert(0, "ordre", range(1, len(trajet) + 1))
print(len(trajet), "arrêts du récit")
trajet[["ordre", "lieu", "debut_pct", "fin_pct", "mentions"]]

## B2. Exporter le trajet, avec la "vitesse narrative"

Pour chaque segment du trajet, on calcule :
- `distance_km` : la distance à vol d'oiseau (grand cercle) entre les deux lieux ;
- `part_du_roman_pct` : la part du texte écoulée entre les deux arrivées ;
- `km_par_pct` : les kilomètres parcourus par % de texte. **Élevé** = le récit va vite, voire saute (ellipse) ; **faible** = le récit s'attarde.

Le segment qui traverse l'antiméridien (180°) est coupé en deux pour ne pas traverser toute la carte.

In [ ]:
from math import asin, cos, radians, sin, sqrt


def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = radians(lat1), radians(lat2)
    dp, dl = p2 - p1, radians(lon2 - lon1)
    return 2 * 6371 * asin(sqrt(sin(dp / 2) ** 2 + cos(p1) * cos(p2) * sin(dl / 2) ** 2))


def segment(lon1, lat1, lon2, lat2):
    # géométrie GeoJSON d'un segment, coupée si elle traverse l'antiméridien
    if abs(lon2 - lon1) <= 180:
        return {"type": "LineString", "coordinates": [[lon1, lat1], [lon2, lat2]]}
    lon2_deroule = lon2 + 360 if lon2 < lon1 else lon2 - 360
    bord = 180 if lon2_deroule > lon1 else -180
    t = (bord - lon1) / (lon2_deroule - lon1)
    lat_bord = lat1 + t * (lat2 - lat1)
    return {"type": "MultiLineString", "coordinates": [
        [[lon1, lat1], [bord, lat_bord]],
        [[-bord, lat_bord], [lon2, lat2]],
    ]}


def trajet_vers_geojson(points, col_position, chemin, colonnes_en_plus=()):
    # un segment par couple de lieux successifs ; renvoie aussi le tableau des segments
    lignes, features = [], []
    recs = points.to_dict("records")
    for a, b in zip(recs, recs[1:]):
        dist = haversine_km(a["lat"], a["lon"], b["lat"], b["lon"])
        part = round(b[col_position] - a[col_position], 1)
        props = {
            "ordre": int(b["ordre"]), "de": a["lieu"], "vers": b["lieu"],
            "depart_pct": float(a[col_position]), "arrivee_pct": float(b[col_position]),
            "position_pct": float(b[col_position]),      # un segment est parcouru à l'arrivée
            "distance_km": round(dist), "part_du_roman_pct": part,
            "km_par_pct": round(dist / part) if part > 0 else None,
        }
        props.update({c: _json_safe(b.get(c)) for c in colonnes_en_plus})
        lignes.append(props)
        features.append({"type": "Feature", "geometry": segment(a["lon"], a["lat"], b["lon"], b["lat"]),
                         "properties": props})
    Path(chemin).write_text(json.dumps({"type": "FeatureCollection", "features": features},
                                       ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"Écrit : {chemin} ({len(features)} segments)")
    return pd.DataFrame(lignes)


segments_recit = trajet_vers_geojson(trajet, "debut_pct", fichier("trajet_recit.geojson"))
points_vers_geojson(trajet.assign(position_pct=trajet["debut_pct"]), fichier("trajet_recit_points.geojson"))
segments_recit.sort_values("km_par_pct", ascending=False).head(10)

## B3. Le récit et l'espace sur un seul graphique

En abscisse, la position dans le texte ; en ordonnée, la longitude du lieu où se trouve le récit. Un palier = le récit s'installe ; une marche brusque = il saute d'un lieu lointain à l'autre. Pour un tour du monde vers l'est, la courbe monte.

In [ ]:
import numpy as np


def longitude_deroulee(lons):
    return np.degrees(np.unwrap(np.radians(np.asarray(lons, dtype=float))))


fig, ax = plt.subplots(figsize=(14, 5))
ax.step(trajet["debut_pct"], longitude_deroulee(trajet["lon"]), where="post", color="#B5472F",
        label="trajet du récit (automatique)")
for _, t in trajet.iterrows():
    ax.annotate(t["lieu"], (t["debut_pct"], longitude_deroulee(trajet["lon"])[t["ordre"] - 1]),
                fontsize=7, xytext=(2, 4), textcoords="offset points")
ax.set(xlabel="position dans le texte (%)", ylabel="longitude (déroulée, °)", xlim=(0, 100))
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
plt.show()

---
# Partie C — Un itinéraire de référence

Si vous avez un itinéraire connu (`FICHIER_ETAPES`), on le **date dans le temps du récit** : la position, en % du texte, où le personnage arrive à chaque étape.

La première mention ne convient pas : dans le *Tour du monde*, tout l'itinéraire est cité dès le Reform Club (vers 6 %). On retient pour chaque étape le passage où son nom est **le plus dense**, avec une contrainte : chaque arrivée vient après la précédente (programmation dynamique). Une étape introuvable reçoit une position interpolée.

C'est une estimation : **vérifiez-la** avec l'extrait affiché, et corrigez au besoin dans `CORRECTIONS_POSITIONS`.

> Le temps du récit n'est pas le temps de l'histoire. Chez Verne, Paris, Turin et Brindisi tombent tous vers 16 % : ces étapes ne sont pas racontées, on les découvre d'un coup à Suez, dans le carnet de Fogg. C'est une **ellipse**, que `km_par_pct` fait ressortir.

In [ ]:
FENETRE = 20   # nombre de paragraphes sur lesquels on mesure la densité d'un lieu


def positions_arrivee(paragraphes, noms, fenetre=FENETRE):
    n = len(paragraphes)
    occurrences = []
    for nom in noms:
        regex = re.compile(motif(nom), flags=re.IGNORECASE)
        occurrences.append([i for i, p in enumerate(paragraphes) if regex.search(p)])

    def densite(hits, p):
        return sum(1 for h in hits if p <= h < p + fenetre)

    etats = {0: (0, [0])}                     # le départ est au début du texte
    for hits in occurrences[1:]:
        nouveaux = {}
        for dernier, (score, chemin) in etats.items():
            options = [(dernier, score, chemin + [None])]          # étape non trouvée
            options += [(p, score + densite(hits, p), chemin + [p]) for p in hits if p > dernier]
            for cle_p, s, ch in options:
                if cle_p not in nouveaux or s > nouveaux[cle_p][0]:
                    nouveaux[cle_p] = (s, ch)
        etats = nouveaux
    _, chemin = max(etats.values(), key=lambda v: v[0])
    pct = pd.Series([None if p is None else p / n * 100 for p in chemin], dtype="float")
    return chemin, pct.interpolate().round(1)


CORRECTIONS_POSITIONS = {
    # ordre de l'étape: position en % du texte, ex. 1: 13.0 (départ réel de Londres)
    # à compléter après vérification
}

if etapes is not None:
    chemin_paragraphes, etapes["arrivee_pct"] = positions_arrivee(paragraphes, etapes["nom_texte"])
    etapes["extrait"] = ["(interpolé)" if p is None else paragraphes[p][:110] + "…" for p in chemin_paragraphes]
    for ordre, pct in CORRECTIONS_POSITIONS.items():
        etapes.loc[etapes["ordre"] == ordre, ["arrivee_pct", "extrait"]] = [pct, "(correction manuelle)"]
    pd.set_option("display.max_colwidth", 120)
    display(etapes[["ordre", "lieu", "arrivee_pct", "extrait"]])
else:
    print("Pas d'itinéraire de référence : partie C sautée.")

In [ ]:
if etapes is not None:
    en_plus = [c for c in ("mode_arrivee", "note") if c in etapes.columns]
    segments_ref = trajet_vers_geojson(etapes, "arrivee_pct", fichier("itineraire.geojson"), en_plus)
    points_vers_geojson(
        etapes.drop(columns=["extrait"]).drop_duplicates("lieu").assign(position_pct=lambda d: d["arrivee_pct"]),
        fichier("etapes.geojson"))

    # comparaison avec le trajet automatique (B3)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.step(trajet["debut_pct"], longitude_deroulee(trajet["lon"]), where="post", color="#B5472F",
            label="trajet du récit (automatique)")
    ax.step(etapes["arrivee_pct"], longitude_deroulee(etapes["lon"]), where="post", color="#2F6F8F",
            label="itinéraire de référence")
    ax.set(xlabel="position dans le texte (%)", ylabel="longitude (déroulée, °)", xlim=(0, 100))
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")
    plt.show()
    display(segments_ref[["de", "vers", "distance_km", "part_du_roman_pct", "km_par_pct"]])

---
# Partie D — Des réseaux pour Gephi

Deux réseaux tirés du texte, exportés au format **GEXF** (format natif de Gephi). Les lieux géocodés portent `lat` et `lon` (nombres décimaux) : le plugin **GeoLayout** peut les placer sur une carte, et **ForceAtlas 2** selon leurs liens. Comparer les deux placements, c'est comparer **géographie réelle** et **géographie du récit**.

## D1. Les lieux cités ensemble

Deux lieux sont liés s'ils apparaissent dans le même passage (`FENETRE_COOC` paragraphes). Le poids du lien = le nombre de passages communs.

In [ ]:
import itertools
from collections import Counter

import networkx as nx

FENETRE_COOC = 1          # 1 = même paragraphe ; 3 = blocs de 3 paragraphes…
MIN_MENTIONS_RESEAU = 2   # on écarte les lieux cités une seule fois
MIN_COOC = 1              # poids minimal d'un lien

cles_reseau = set(lieux_propres.loc[lieux_propres["nb_mentions"] >= MIN_MENTIONS_RESEAU, "lieu"].map(cle))
m_res = mentions[mentions["cle"].isin(cles_reseau)].assign(bloc=lambda d: d["paragraphe"] // FENETRE_COOC)
lieux_par_bloc = m_res.groupby("bloc")["cle"].agg(lambda s: sorted(set(s)))

paires = Counter(p for cles in lieux_par_bloc for p in itertools.combinations(cles, 2))

infos = lieux_propres.assign(cle=lieux_propres["lieu"].map(cle)).set_index("cle")
coords = lieux_geo.set_index("cle")


def attributs_lieu(k):
    d = {"label": infos.at[k, "lieu"], "type": "lieu",
         "nb_mentions": int(infos.at[k, "nb_mentions"]),
         "premiere_mention_pct": float(infos.at[k, "premiere_mention_pct"])}
    if k in coords.index:
        d.update(lat=float(coords.at[k, "lat"]), lon=float(coords.at[k, "lon"]),
                 type_lieu=str(coords.at[k, "type_lieu"]))
    return d


G = nx.Graph()
for (a, b), w in paires.items():
    if w >= MIN_COOC:
        G.add_edge(a, b, weight=w)
for k in G.nodes:
    G.nodes[k].update(attributs_lieu(k))

nx.write_gexf(G, fichier("reseau_lieux.gexf"))
nb_geo = sum(1 for _, d in G.nodes(data=True) if "lat" in d)
print(f"{G.number_of_nodes()} lieux ({nb_geo} géolocalisés), {G.number_of_edges()} liens -> {fichier('reseau_lieux.gexf')}")
pd.DataFrame([(G.nodes[a]["label"], G.nodes[b]["label"], d["weight"]) for a, b, d in G.edges(data=True)],
             columns=["lieu 1", "lieu 2", "passages communs"]).sort_values("passages communs", ascending=False).head(10)

## D2. Personnages et lieux

Un réseau **biparti** : d'un côté les personnages (entités `PER` / `PERSON`), de l'autre les lieux ; un lien quand ils apparaissent dans le même passage. On voit qui est associé à quels lieux.

**À vous** : regroupez les variantes d'un même personnage dans `FUSIONS_PERSONNAGES`, et retirez les faux personnages avec `PAS_DES_PERSONNAGES`.

In [ ]:
PERSONNAGES = {
    # personnage : [les noms qui le désignent dans le texte]   (exemples pour le Tour du monde)
    "Fogg": ["Phileas Fogg", "Mr. Fogg", "Fogg"],
    "Passepartout": ["Passepartout"],
    "Fix": ["Fix"],
    "Aouda": ["Mrs. Aouda", "Aouda"],
    "Francis Cromarty": ["Francis Cromarty", "Cromarty"],
    "Proctor": ["Stamp W. Proctor", "Proctor"],
    # à compléter / remplacer pour votre texte
}
N_PERSONNAGES = 12          # utilisé seulement si aucun nom de PERSONNAGES n'est trouvé
PAS_DES_PERSONNAGES = {
    # lieux ou mots pris pour des personnes
}

label_per = "PER" if LANGUE == "fr" else "PERSON"
candidats = (
    entites[entites["label"] == label_per]["texte"].map(normaliser)
    .loc[lambda s: ~s.map(cle).isin(set(infos.index)) & ~s.isin(PAS_DES_PERSONNAGES)]
    .value_counts()
)
print("Noms proposés par spaCy :", ", ".join(f"{n} ({c})" for n, c in candidats.head(20).items()))


def motif_personnage(noms):
    variantes = sorted(set(noms), key=len, reverse=True)
    return re.compile(r"\b(?:" + "|".join(re.escape(v) for v in variantes) + r")\b")


def compter(personnages):
    lignes = []
    for nom, variantes in personnages.items():
        rx = motif_personnage(variantes)
        for i, p in enumerate(paragraphes):
            n = len(rx.findall(p))
            if n:
                lignes.append((nom, i, n))
    return pd.DataFrame(lignes, columns=["nom", "paragraphe", "n"])


occ = compter(PERSONNAGES)
if occ.empty:
    print("Aucun nom de PERSONNAGES dans ce texte : on prend les noms PER les plus fréquents.")
    occ = compter({n: [n] for n in candidats.head(N_PERSONNAGES).index})

nb_mentions = occ.groupby("nom")["n"].sum().sort_values(ascending=False)
absents = [n for n in PERSONNAGES if n not in nb_mentions.index]
if absents and len(absents) < len(PERSONNAGES):
    print("Introuvables dans le texte :", ", ".join(absents))
print("Personnages retenus :", ", ".join(f"{n} ({c})" for n, c in nb_mentions.items()))

occ["bloc"] = occ["paragraphe"] // FENETRE_COOC
liens = Counter(
    (p, l)
    for bloc, persos in occ.groupby("bloc")["nom"].agg(set).items()
    if bloc in lieux_par_bloc.index
    for p in persos for l in lieux_par_bloc[bloc]
)

B = nx.Graph()
for (p, l), w in liens.items():
    B.add_edge(f"P:{p}", l, weight=w)
for n in B.nodes:
    if n.startswith("P:"):
        B.nodes[n].update(label=n[2:], type="personnage", nb_mentions=int(nb_mentions[n[2:]]))
    else:
        B.nodes[n].update(attributs_lieu(n))

nx.write_gexf(B, fichier("reseau_personnages_lieux.gexf"))
print(f"{B.number_of_nodes()} nœuds, {B.number_of_edges()} liens -> {fichier('reseau_personnages_lieux.gexf')}")

---
# Partie E — Lieux traversés, lieux évoqués

Tous les lieux cités ne sont pas visités : certains sont seulement évoqués (souvenirs, comparaisons, rêves, nouvelles). On calcule pour chaque lieu sa distance à l'arrêt le plus proche du trajet (l'itinéraire de référence s'il existe, sinon le trajet du récit), et on le classe **traversé** ou **évoqué** selon un seuil en kilomètres.

In [ ]:
SEUIL_KM = 300

reference = etapes if etapes is not None else trajet
arrets = list(zip(reference["lat"], reference["lon"]))
lieux_geo["distance_trajet_km"] = [
    round(min(haversine_km(la, lo, a, b) for a, b in arrets)) for la, lo in zip(lieux_geo["lat"], lieux_geo["lon"])
]
lieux_geo["statut"] = np.where(lieux_geo["distance_trajet_km"] <= SEUIL_KM, "traversé", "évoqué")
exporter_lieux()

print(lieux_geo["statut"].value_counts().to_string())
lieux_geo[lieux_geo["statut"] == "évoqué"].sort_values("nb_mentions", ascending=False)[
    ["lieu", "nb_mentions", "distance_trajet_km", "type_lieu"]].head(15)

---
## Récapitulatif des fichiers produits

| Fichier (préfixé par `NOM_PROJET`) | Contenu | Outil |
|---|---|---|
| `_lieux.csv` / `.geojson` | lieux géocodés : mentions, position, type, distance au trajet, statut | CARTO, QGIS |
| `_trajet_recit.geojson` | trajet du récit, segments avec vitesse narrative | CARTO, QGIS |
| `_trajet_recit_points.geojson` | arrêts du récit | CARTO, QGIS |
| `_itineraire.geojson` / `_etapes.geojson` | itinéraire de référence daté dans le récit *(si fourni)* | CARTO, QGIS |
| `_reseau_lieux.gexf` | lieux cités ensemble | Gephi |
| `_reseau_personnages_lieux.gexf` | personnages et lieux | Gephi |

Toutes les couches cartographiques ont une colonne `position_pct` : dans CARTO, **un seul Range widget** sur cette colonne filtre toutes les couches à la fois (20 %, 50 %, 80 % du texte…).

In [ ]:
for f in sorted(DOSSIER.glob(f"{NOM_PROJET}_*")):
    print(f"{f.name:45} {f.stat().st_size / 1024:8.1f} Ko")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/donnees', 'zip', '/content/donnees')

files.download('/content/donnees.zip')